# Analisi della varianza
Ipotizziamo di avere a disposizione delle osservazioni di un medesimo attributo divise in $G$ gruppi, per esempio perchè si tratta del reddito di individui che vivono in diverse città, oppure di un valore ematico di pazienti sottoposti a diversi trattamenti clinici e così via. Formalmente, indichiamo rispettivamente con $n_1, \dots, n_G$ le numerosità dei vari gruppi, con $n = n_1 + \dots + n_G$ il numero totale di osservazioni e, fissato $g \in \{1, \dots, G\}$ e $i \in \{1, \dots, n_g\}$, denotiamo con $x_i^g$ il valore dell'$i$-esima osservazione nel gruppo $g$

Se si è interessati a valutare l'ipotesi che i valori delle medie nei vari gruppi siano sensibilmente differenti, per esempio perchè si vuole dimostrare che il reddito non sia troppo diverso in un gruppo di città, oppure per dimostrare l'efficiacia di un dato trattamento medico, è possibile applicare un metodo chiamato _ANOVA_ (_ANalysis Of VAriance_). L'idea alla base di questo metodo è che se non vi sono sostanziali differenze tra i gruppi considerati, allora calcolare la varianza all'interno di un gruppo qualsiasi non dovrebbe portare a un risultato molto dissimile da quello ottenuto effettuando il calcolo su tutti i dati a disposizione. Più formalmente, definite:
- la media campionaria $\bar{x} = \frac{1}{n} \sum_{g = 1}^{G} \sum_{i = 1}^{n_g} x_i^g$ su tutte le osservazioni;
- la media campionaria $\bar{x}^g =\frac{1}{n_g} \sum_{i = 1}^{n_g} x_i^g$ all'interno del $g$-esimo gruppo, per ogni $g = 1, \dots, n_G$;
- la somma totale degli scarti $$SS_T = \sum_{g = 1}^{G} \sum_{i = 1}^{n_g} {(x_i^g - \bar{x})}^2$$;
- la somma degli scarti _entro i gruppi_ (o _within groups_) $$SS_W = \sum_{g = 1}^{G} \sum_{i = 1}^{n_g} {(x_i^g - \bar{x}^g)}^2$$;
- la somma degli scarti _tra i gruppi_ (o _between groups_), pesata rispetto alla numerosità dei vari gruppi: $$SS_B = \sum_{g = 1}^{G} n_g {(\bar{x}^g - \bar{x})}^2$$;

A partire da ognuna di queste somme è facile calcolare le corrispondenti varianze campionarie:
- la varianza campionaria su tutte le osservazioni: $s_T^2 = frac{1}{n - 1} SS_T$;
- la varianza campionaria delle medie tra i gruppi: $s_B^2 = \frac{1}{G - 1} SS_B$ (il motivo per cui viene fatta la divisione per $G - 1$ è analoga alla ragione per cui il calcolo della varianza campionaria viene fatta dividendo per $n - 1$);
- la varianza campionaria dei valori entro i gruppi: $s_W^2 = \\frac{1}{n - G} SS_W$

Si può mostrare che $SS_T = SS_W + SS_B$, e quindi che
$$\frac{SS_T}{n - 1} = \frac{SS_W}{n - 1} + \frac{SS_B}{n - 1} = \frac{n - G}{n - 1} \frac{SS_W}{n - G} + \frac{G - 1}{n - 1} \frac{SS_B}{G - 1}$$

Pertanto, 
$$\text{varianza totale} = \frac{n - G}{n - 1} \space \text{varianza entro i gruppi} + \frac{G - 1}{n - 1} \space \text{varianza tra i gruppi}$$

Possiamo utilizzare questa uguaglianza per validare o confutare l'ipotesi che le medie nei gruppi siano diverse: se la varianza totale e la varianza entro i gruppi assumono valori non troppo diversi tra loro (e dunque la varianza tra i gruppi risulta trascurabile), allora si confuta l'ipotesi; se al contrario la varianza tra i gruppi assume un valore elevato si può convalidare l'ipotesi

In realtà per validare o confutare questi tipi di ipotesi esistono metodi quantitativi basati sul calcolo di indici che coinvolgono le quantità che noi abbiamo confrontato in modo qualitativo
Per applicare la tecnica di analisi della varianza selezioniamo per esempio i due gruppi dei supereroi corrispondenti alle due case editrici più rappresentate: Marvel e DC, e concentriamoci per ogni gruppo sull'indice di forza

In [1]:
import pandas as pd

heroes = pd.read_csv('data/heroes.csv', sep=';', index_col=0)

marvel_strength = heroes[(heroes['Publisher'] == 'Marvel Comics') & 
    (pd.notnull(heroes['Strength']))]['Strength']
dc_strength = heroes[(heroes['Publisher'] == 'DC Comics') &
    (pd.notnull(heroes['Strength']))]['Strength']

<div class="alert alert-info">
    La cella precedente mostra come effettuare un'operazione di selezione complessa su di un dataframe, basandosi sull'operatore <code>&</code> di congiunzione logica. Analogamente, l'operatore <code> | </code> permette di calcolare disgiunzioni logiche
</div>

Iniziamo calcolando $SS_T$

In [2]:
all_strength = pd.concat([marvel_strength, dc_strength])
sum_total = sum((all_strength - all_strength.mean())**2)
sum_total

561319.438976378

Analogamente calcoliamo $SS_W$

In [4]:
sum_within = sum((marvel_strength - marvel_strength.mean())**2) + sum((dc_strength - dc_strength.mean())**2)
sum_within

561127.6224760013

Infine, calcoliamo $SS_B$

In [5]:
sum_between = len(marvel_strength) * (marvel_strength.mean() - all_strength.mean())**2 + len(dc_strength) * (dc_strength.mean() - all_strength.mean())**2
sum_between

np.float64(191.81650037662837)

Verifichiamo innanzitutto che valga, nei limiti dell'approssimazione in virgola mobile, l'uguaglianza $SS_T = SS_W + SS_B$

In [6]:
sum_total - sum_within - sum_between

np.float64(4.8771653382573277e-11)

Calcoliamo infine la varianza totale e la varianza entro i gruppi utilizzando le formule sopra descritte

In [7]:
n = len(all_strength)

total_var = sum_total / (n-1)
total_var

1107.1389328922642

In [8]:
within_var = sum_within / (n-2)
within_var * (n-2) / (n-1)

1106.760596599608

I due valori sono molto vicini, e quindi possiamo avvalorare l'ipotesi che i valori medi dell'indice di forza siano sostanzialmente uguali

Riscriviamo il codice in modo da incorporare in una funzione la procedura di analisi della varianza, in modo da gestire anche più di due gruppi di osservazioni: la funzione accetterà una lista di tali gruppi come argomento, e restituirà una coppia i cui elementi saranno rispettivamente la varianza totale e la varianza entro i gruppi

In [9]:
import numpy as np

def anova(groups):
    all_elements = pd.concat(groups)

    sum_total = sum ((all_elements - all_elements.mean())**2)
    sum_within = sum([sum((g - g.mean())**2) for g in groups])

    sum_between = sum([len(g) * (g.mean()-all_elements.mean())**2 for g in groups])

    assert(np.abs(sum_total - sum_within - sum_between) < 10**-5)
    n = len(all_elements)
    total_var = sum_total / (n-1)
    within_var = sum_within / (n-len(groups))

    return (total_var, within_var*(n-len(groups))/(n-1))

Verifichiamo che i valori restituiti per i due gruppi già considerati siano gli stessi

In [10]:
anova([dc_strength, marvel_strength])

(1107.1389328922642, 1106.760596599608)

Applicando la procedura all'indice di forza suddiviso tra supereroi e supereroine si ottengono due valori tutto sommato relativamente simili, così che non si possa avvalorare l'ipotesi che l'indice di forza sia distribuito in modo sostanzialmente diverso tra i due generi

In [12]:
male_strength = heroes[(heroes['Gender'] == 'M') & (pd.notnull(heroes['Strength']))]['Strength']
female_strength = heroes[(heroes['Gender'] == 'F') & (pd.notnull(heroes['Strength']))]['Strength']
anova([male_strength, female_strength])

(1083.4610943430741, 1072.1108493522433)

Le cose cambiano se consideriamo la divisione tra generi per i supereroi DC, valutando la differenza nella distribuzione del peso

In [13]:
male_weight = heroes[(heroes['Publisher'] == 'DC Comics') & (heroes['Gender'] == 'M') & (pd.notnull(heroes['Weight']))]['Weight']
female_weight = heroes[(heroes['Publisher'] == 'DC Comics') & (heroes['Gender'] == 'F') & (pd.notnull(heroes['Weight']))]['Weight']
anova([male_weight, female_weight])

(9314.588141839082, 9049.012630905872)